# Deber — Semana 3: Carga de Datos, Series/DataFrames, loc/iloc y Análisis de Texto
## Análisis de Datos (TDSD353) | Período 2026-A
## ESFOT — Escuela Politécnica Nacional


**Nombre:** Haziel Moncayo
<br/>
**Fecha:** 27/04/2026
<br/>
---

### Objetivo de este Taller
Aplicar las técnicas aprendidas en clase para:
1. Cargar datos desde archivos **JSON**, **TXT** (CSV) y **XML** usando Pandas.
2. Detectar y documentar **problemas de calidad de datos**.
3. Unificar múltiples DataFrames con `pd.concat()` y exportar a CSV.
4. Comprender y usar **Series** y **DataFrames**.
5. Acceder a datos con **`loc`** (por etiqueta) e **`iloc`** (por posición).
6. Descargar y procesar texto desde la web con `urllib`.

### Archivos de trabajo
| Archivo | Contenido | IDs |
|---------|-----------|-----|
| `productos.json` | 20 productos de librería | 1–20 |
| `productos.txt` | 20 productos de librería (formato CSV) | 21–40 |
| `productos.xml` | 20 productos de librería | 41–60 |
| `netflix.csv` | Catálogo de Netflix (~8800 títulos) | — |

> ⚠️ Los archivos de productos contienen **errores intencionales** (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

---

---
## PARTE A: Pipeline de datos — Inventario de Papelería

Tenemos 3 archivos con datos del inventario de una Papelería:
- `productos.json` — 20 productos (IDs 1–20)
- `productos.txt` — 20 productos (IDs 21–40, separados por coma)
- `productos.xml` — 20 productos (IDs 41–60)

Columnas compartidas: `id`, `nombre`, `precio_compra`, `categoria`, `stock`, `proveedor`, `precio_venta_publico`

> ⚠️ Los datos contienen errores intencionales (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

### A-1. Importar librerías

| Librería | Uso |
|----------|-----|
| `pandas` | Manipulación de datos tabulares |
| `urllib.request` | Descarga de archivos desde la web |
| `collections.Counter` | Conteo de frecuencias |
| `time` | Medición de tiempos |
| `re` | Expresiones regulares para limpieza de texto |

In [54]:
import pandas as pd
import urllib.request
from collections import Counter
import time 
import re

### A-2. Cargar JSON con `pd.read_json()`

**JSON** (JavaScript Object Notation): formato ligero muy común en APIs web.

```python
df = pd.read_json('archivo.json')
```

Parámetros útiles: `orient` (formato del JSON), `encoding`, `dtype` (forzar tipos).

In [56]:
json = pd.read_json('productos.json')

### A-3. Cargar TXT con `pd.read_csv()`

**TXT tabular** (datos separados por coma): los archivos `.txt` con formato de tabla se leen con `read_csv()` ajustando el separador.

```python
df = pd.read_csv('archivo.txt', sep=',')   # separador: coma
df = pd.read_csv('archivo.txt', sep='\t')  # separador: tabulación
```

> **Diferencia CSV vs TXT**: ambos son texto plano. La extensión `.txt` no implica un formato fijo — siempre revisa el separador antes de cargar.

In [57]:
txt = pd.read_csv('productos.txt')

### A-4. Cargar XML con `pd.read_xml()`

**XML** (eXtensible Markup Language): formato jerárquico usado en servicios web SOAP y datos gubernamentales.

```python
df = pd.read_xml('archivo.xml', xpath='.//producto')
```

El parámetro `xpath` indica qué nodos representan cada fila. 

In [58]:
xml = pd.read_xml('productos.xml', xpath = './/producto')

### A-5. Verificar consistencia y detectar errores

Antes de concatenar, verificamos que los 3 DataFrames tengan las **mismas columnas** y documentamos los problemas de calidad.

Errores comunes en datos reales:
- Valores `null` / `NaN` (datos faltantes)
- Texto donde debería haber números (`"error"`, `"N/A"`, `"texto"`)
- Valores lógicamente inválidos (Ejemplo stock negativo)
- Campos vacíos (`""`)

In [59]:
# .isna() identifica los NaN y .sum() los cuenta por columna
print("-- JSON --")
print("--- Datos faltantes en JSON ---")
print(json.isna().sum())

print("--- Conteo de palabras de error ---")
print(json.isin(['error', 'N/A', 'texto']).sum())
print("--- Conteo de celdas vacías ---")
print((json == "").sum())
# Convertimos stock a número; si hay texto, se vuelve NaN temporalmente
stock_numerico = pd.to_numeric(json['stock'], errors='coerce')
print("--- Cantidad de productos con stock negativo ---")
print((stock_numerico < 0).sum())

-- JSON --
--- Datos faltantes en JSON ---
id                      0
nombre                  0
precio_compra           1
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64
--- Conteo de palabras de error ---
id                      0
nombre                  0
precio_compra           1
categoria               0
stock                   1
proveedor               0
precio_venta_publico    0
dtype: int64
--- Conteo de celdas vacías ---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
--- Cantidad de productos con stock negativo ---
1


In [60]:
print("-- TXT --")
print("--- Datos faltantes en TXT ---")
print(txt.isna().sum())

print("\n--- Conteo de palabras de error ---")
print(txt.isin(['error', 'N/A', 'texto']).sum())

print("\n--- Conteo de celdas vacías ---")
print((txt == "").sum())

# Manejamos el stock igual que antes por si hay textos mezclados
stock_num_txt = pd.to_numeric(txt['stock'], errors='coerce')
print("\n--- Cantidad de productos con stock negativo ---")
print((stock_num_txt < 0).sum())

-- TXT --
--- Datos faltantes en TXT ---
id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64

--- Conteo de palabras de error ---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    1
dtype: int64

--- Conteo de celdas vacías ---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64

--- Cantidad de productos con stock negativo ---
0


In [64]:
print("-- XML --")
print("--- Datos faltantes en XML ---")
print(xml.isna().sum())

print("\n--- Conteo de palabras de error ---")
print(xml.isin(['error', 'N/A', 'texto']).sum())

print("\n--- Conteo de celdas vacías ---")
print((xml == "").sum())

# Manejamos el stock con to_numeric por si hay textos mezclados
stock_num_xml = pd.to_numeric(xml['stock'], errors='coerce')
print("\n--- Cantidad de productos con stock negativo ---")
print((stock_num_xml < 0).sum())

-- XML --
--- Datos faltantes en XML ---
id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64

--- Conteo de palabras de error ---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   1
proveedor               0
precio_venta_publico    1
dtype: int64

--- Conteo de celdas vacías ---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64

--- Cantidad de productos con stock negativo ---
1


### A-6. Concatenar con `pd.concat()` y exportar con el metodo to_csv()

`pd.concat()` apila DataFrames verticalmente. `ignore_index=True` reinicia el índice.

| Función | Uso |
|---------|-----|
| `pd.concat()` | Apilar DataFrames con **misma estructura** |

In [65]:
df_total = pd.concat([json, txt, xml], ignore_index=True)
# Verificacion
print(f"Cantidad total de productos: {df_total.shape[0]}")
df_total

Cantidad total de productos: 60


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300,Norma Ecuador,3.5
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150,OfficeMax Ecuador,1.2
4,5,Carpeta Manila A4,0.15,Archivo,1000,PapelPro Cía. Ltda.,0.3
5,6,Tijeras Escolares 13cm,0.9,Manualidades,200,OfficeMax Ecuador,1.8
6,7,Corrector Líquido 18ml,None,Escritura,180,Norma Ecuador,1.5
7,8,Goma de Borrar Blanca,0.25,Escritura,600,DistribuPapel S.A.,0.55
8,9,Resma Papel Bond A4 75g,3.2,Papel,250,PapelPro Cía. Ltda.,5.5
9,10,Marcador Permanente Negro,0.6,Escritura,400,OfficeMax Ecuador,1.1


Exportar el dataframe concatenado con `to_csv()` con index=False  como argumento adicional, evitar escribir el índice numérico como columna extra

In [67]:
# Guardamos el resultado en un archivo nuevo
df_total.to_csv('productos_final.csv', index=False)

print("Guardado exitosamente")

Guardado exitosamente


---
---
## PARTE B: Series, DataFrames, `loc` e `iloc` — Catálogo de Netflix

Trabajaremos con el archivo `netflix.csv`, un dataset real con el catálogo de contenidos de Netflix.

| Columna | Descripción |
|---------|------------|
| `show_id` | ID único del título |
| `type` | `Movie` o `TV Show` |
| `title` | Nombre del título |
| `director` | Director(es) |
| `cast` | Elenco principal |
| `country` | País de producción |
| `date_added` | Fecha de añadido a Netflix |
| `release_year` | Año de estreno |
| `rating` | Clasificación de audiencia |
| `duration` | Duración (minutos o temporadas) |
| `listed_in` | Géneros/categorías |
| `description` | Sinopsis |

### B-1. Carga el dataset y realiza una primera inspección

In [68]:
netflix = pd.read_csv("netflix.csv")
netflix.head(3)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN
1,tm82169,Rocky,MOVIE,"When world heavyweight boxing champion, Apollo...",1976,PG,119,"['drama', 'sport']",['US'],NaN,tt0075148,8.1,588100.0,106.361,7.782
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406


---
### B-2. ¿Qué es una Series?

Una **Series** es un arreglo **unidimensional** etiquetado. Cada elemento tiene un **valor** y un **índice**.

```python
# Desde una lista
s = pd.Series([10, 20, 30])

# Desde un diccionario (claves = índice)
s = pd.Series({'a': 10, 'b': 20, 'c': 30})
```

| Atributo | Descripción |
|----------|------------|
| `.values` | Datos como array NumPy |
| `.index` | Etiquetas del índice |
| `.dtype` | Tipo de dato |
| `.name` | Nombre de la Series |
| `.shape` | Forma (número de elementos,) |

**Cuando accedes a UNA columna de un DataFrame, obtienes una Series:**

### Extraer una columna = una Series del csv de netflix

In [69]:
col_net = netflix['title']
col_net

0       Five Came Back: The Reference Films
1                                     Rocky
2                                    Grease
3                                 The Sting
4                                  Rocky II
                       ...                 
6132                          عبود في البيت
6133                                Sweetie
6134              Sommore: Queen Chandelier
6135                           All Na Vibes
6136                        Going to Heaven
Name: title, Length: 6137, dtype: str

### Operaciones comunes sobre Series

### Obten las estadísticas de la columna 'release_year', count, min, max, mean, median, std

In [70]:
años = netflix['release_year']
print(f"Conteo (No nulos): {años.count()}")
print(f"Año más antiguo: {años.min()}")
print(f"Año mas reciente: {años.max()}")
print(f"Promedio de años: {años.mean()}")
print(f"Año central (median): {años.median()}")
print(f"Desviacion estanar: {años.std()}")
print()
print(f"Metodo nuevo investigado (años.describe()): {años.describe()}")

Conteo (No nulos): 6137
Año más antiguo: 1945
Año mas reciente: 2023
Promedio de años: 2017.3718429199935
Año central (median): 2019.0
Desviacion estanar: 6.603620140967784

Metodo nuevo investigado (años.describe()): count    6137.000000
mean     2017.371843
std         6.603620
min      1945.000000
25%      2017.000000
50%      2019.000000
75%      2021.000000
max      2023.000000
Name: release_year, dtype: float64


### Obten el Top 10 países con más contenido producido

In [71]:
conteo_paises = netflix['production_countries'].value_counts()

top_10_paises = conteo_paises.iloc[0:10]

print("Top 10 países:")
print(top_10_paises)

Top 10 países:
production_countries
['US']    1981
['IN']     633
['JP']     278
['KR']     259
['GB']     235
['ES']     177
[]         176
['FR']     121
['MX']     115
['BR']     104
Name: count, dtype: int64


### Obten el Top 10 de series/peliculas mas vistas

In [72]:
netflix_ordenado = netflix.sort_values(by='tmdb_popularity', ascending=False)

# 2. Usamos iloc para las primeras 10 filas y las columnas 0 y 1
# (Asumiendo que la columna 0 es 'title' y la 1 es la popularidad)
top_10_vistas = netflix_ordenado.iloc[0:10, [0, 1]]

print("Top 10 más vistas:")
print(top_10_vistas)

Top 10 más vistas:
             id                     title
675     ts11188                 The Flash
6049   ts375464                  Triptych
188     ts21469            Grey's Anatomy
4258   ts274631                 Wednesday
4291   tm823609                    Narvik
4993  tm1278737               Lesson Plan
187         ts9          The Walking Dead
6068   ts264903  The Lying Life of Adults
6063  tm1158764                    JUNG_E
676      ts1639             The Blacklist


---
### B-3. Acceso con `loc` (por etiqueta)

**`loc`** selecciona datos por **etiqueta** del índice y **nombre** de columna.

```python
df.loc[etiqueta_fila]                    # Una fila completa
df.loc[etiqueta_fila, 'columna']         # Un valor específico
df.loc[inicio:fin]                       # Rango de filas (INCLUSIVO)
df.loc[condicion]                        # Filtrado booleano
df.loc[condicion, ['col1', 'col2']]      # Filtrado + columnas
```

> Con `loc`, los rangos son **INCLUSIVOS** en ambos extremos: `loc[2:5]` incluye 2, 3, 4 **y 5**.

In [73]:
# Genera un ejemplo de Acceso a una fila completa por su etiqueta de índice
fila_ejemplo = netflix.loc[[0]]
fila_ejemplo

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN


In [74]:
# Filtrado con MÚLTIPLES condiciones
# Obten el listado de Películas de Estados Unidos producidas desde 2018
netflix.loc[(netflix['production_countries'] == "['US']" ) & (netflix['release_year'] >= 2018)]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1422,ts79813,You,SHOW,"A dangerously charming, intensely obsessive yo...",2018,TV-MA,48,"['drama', 'romance', 'thriller', 'crime']",['US'],4.0,tt7335184,7.7,261831.0,341.151,8.118
1423,ts81927,New Amsterdam,SHOW,The new medical director breaks the rules to h...,2018,TV-14,43,['drama'],['US'],5.0,tt7817340,8.0,43464.0,163.879,8.462
1428,ts78801,Cobra Kai,SHOW,This Karate Kid sequel series picks up 30 year...,2018,TV-14,33,"['action', 'drama', 'comedy', 'sport']",['US'],6.0,tt7221388,8.5,190785.0,218.009,8.200
1431,ts81294,Manifest,SHOW,After landing from a turbulent but routine fli...,2018,TV-14,42,"['scifi', 'drama', 'thriller']",['US'],4.0,tt8421350,7.1,71203.0,176.155,7.733
1432,ts83970,All American,SHOW,When a rising high school football player from...,2018,TV-14,43,"['drama', 'sport']",['US'],5.0,tt11574984,7.6,9704.0,75.706,8.209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6098,ts308560,African Queens: Njinga,SHOW,Expert interviews and other documentary conten...,2023,TV-14,47,"['history', 'documentation']",['US'],2.0,tt15305648,5.5,361.0,6.023,NaN
6109,ts367818,Bling Empire: New York,SHOW,"Meet a fresh group of wealthy, sophisticated a...",2023,TV-MA,40,['reality'],['US'],1.0,tt22481904,5.2,376.0,6.621,NaN
6122,ts364674,Princess Power,SHOW,Princess friends from four different Fruitdoms...,2023,TV-Y,15,"['family', 'fantasy', 'animation', 'comedy']",['US'],1.0,tt22013036,7.0,60.0,6.319,8.500
6126,tm1291873,Andrew Santino: Cheeseburger,MOVIE,No topic is safe in this unfiltered stand-up s...,2023,NaN,58,"['comedy', 'documentation']",['US'],NaN,tt24640480,6.5,808.0,3.181,5.500


In [75]:
# Filtrado con isin() para múltiples valores
# Filtra todas las filas cuyo Contenido tenga como origen Latinoamérica
paises_latam = ['Mexico', 'Argentina', 'Colombia', 'Chile', 'Brazil', 'Peru']
latam_df = netflix.loc[netflix['production_countries'].isin(paises_latam)]
latam_df

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score


In [76]:
# str.contains(): filtrado por texto parcial en columna
# Títulos que contengan 'Love' en el nombre
love_df = netflix.loc[netflix['title'].str.contains('Love', case=False, na=False)]

love_df

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
299,tm97072,Eat Pray Love,MOVIE,Liz Gilbert had everything a modern woman is s...,2010,PG-13,133,"['drama', 'romance']",['US'],NaN,tt0879870,5.8,100026.0,19.052,6.230
322,tm464888,Love Aaj Kal,MOVIE,When professional ambitions clash with persona...,2009,PG-13,141,"['drama', 'comedy', 'romance']",['IN'],NaN,tt1275863,6.8,16443.0,3.097,5.500
418,ts4866,Fated to Love You,SHOW,"Fated to Love You, also known as You're My Des...",2008,TV-G,67,"['drama', 'comedy', 'romance']",['TW'],1.0,tt1483930,7.4,510.0,16.052,6.833
441,tm95434,Love in a Puff,MOVIE,When the Hong Kong government enacts a ban on ...,2010,NaN,104,"['comedy', 'drama', 'romance']",['HK'],NaN,tt1602479,7.1,2746.0,4.220,6.953
447,tm76399,A Love Story,MOVIE,Ian Montes is a picture of success. Despite be...,2007,NaN,117,"['romance', 'drama']",['PH'],NaN,tt0990433,6.5,150.0,1.410,4.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6076,ts360024,In Love All Over Again,SHOW,"Year 2003, Irene a young film student who is p...",2023,TV-MA,45,"['drama', 'romance', 'comedy']",['ES'],1.0,tt21206964,6.3,669.0,85.435,6.400
6078,tm1304302,Love at First Kiss,MOVIE,"The story of Javier who, at the age of 16, whi...",2023,NaN,96,"['comedy', 'romance']",['ES'],NaN,tt14463514,5.7,897.0,152.671,6.246
6100,tm1300541,Squared Love All Over Again,MOVIE,The relationship of a well-known journalist an...,2023,NaN,99,"['comedy', 'romance']",['PL'],NaN,tt26369131,4.5,383.0,25.780,5.600
6101,ts314687,Love to Hate You,SHOW,"For a woman who despises losing to men, and a ...",2023,TV-MA,54,"['comedy', 'drama', 'romance']",['KR'],1.0,tt26229508,7.9,1584.0,36.117,8.300


### B-4. Ya que tienes acceso a los datos de Netflix, en base a tu curiosidad, usando pandas obten al menos 3 datos de interes adicionales, que te gustaria conocer.

In [77]:
print("Distribución de contenido en Netflix:")
netflix['type'].value_counts()

Distribución de contenido en Netflix:


type
MOVIE    3831
SHOW     2306
Name: count, dtype: int64

In [78]:
top_calificados = netflix.nlargest(5, 'imdb_score')[['title', 'imdb_score', 'release_year']]

print("Las 5 joyas de Netflix según IMDb:")
top_calificados

Las 5 joyas de Netflix según IMDb:


,title,imdb_score,release_year
2343,Crazy Delicious,9.6,2017
2634,#ABtalks,9.6,2018
186,Breaking Bad,9.5,2008
529,Khawatir,9.5,2005
192,Avatar: The Last Airbender,9.3,2005


In [79]:
print("Promedio de duración (en minutos):")
netflix.groupby('type')['runtime'].mean()

Promedio de duración (en minutos):


type
MOVIE    98.665884
SHOW     39.361232
Name: runtime, dtype: float64

---
### B-5. Acceso con `iloc` (por posición entera)

**`iloc`** selecciona datos por **posición numérica** (como índices de listas en Python).

```python
df.iloc[posicion]                          # Una fila
df.iloc[pos_fila, pos_columna]             # Un valor
df.iloc[inicio:fin]                        # Rango (EXCLUSIVO al final)
df.iloc[[0, 5, 10]]                        # Posiciones específicas
df.iloc[::n]                               # Cada n filas
```

> Con `iloc`, el rango es **EXCLUSIVO** al final: `iloc[2:5]` incluye posiciones 2, 3, 4 pero **NO 5**.

### Diferencia clave `loc` vs `iloc`:

| | `loc` | `iloc` |
|--|-------|--------|
| **Acceso** | Por **etiqueta** | Por **posición** |
| **Rango** | **Inclusivo** | **Exclusivo** |
| **Columnas** | Por nombre | Por número (0, 1, 2...) |
| **Filtrado** |  Booleano | No directo |

In [80]:
# Obten la Primera fila (posición 0) usando loc
primera_fila = netflix.loc[0]
primera_fila

id                                                               ts300399
title                                 Five Came Back: The Reference Films
type                                                                 SHOW
description             This collection includes 12 World War II-era p...
release_year                                                         1945
age_certification                                                   TV-MA
runtime                                                                51
genres                                                  ['documentation']
production_countries                                               ['US']
seasons                                                               1.0
imdb_id                                                               NaN
imdb_score                                                            NaN
imdb_votes                                                            NaN
tmdb_popularity                       

In [81]:
# Última fila (índice negativo, como listas de Python) usando loc
ultima_fila = netflix.loc[netflix.index[-1]]
ultima_fila

id                                                               tm561709
title                                                     Going to Heaven
type                                                                MOVIE
description             A story about young boy sultan, 11 years old l...
release_year                                                         2023
age_certification                                                     NaN
runtime                                                                90
genres                                                         ['family']
production_countries                                               ['AE']
seasons                                                               NaN
imdb_id                                                         tt4623222
imdb_score                                                            7.0
imdb_votes                                                           40.0
tmdb_popularity                       

In [82]:
# Valor específico: fila 10, columna 2 (title) usando loc
valor_especifico = netflix.loc[10, 'title']
valor_especifico

'Heroes'

In [83]:
# Rango de filas (EXCLUSIVO: posiciones 2, 3, 4 — NO incluye 5) usando loc
rango_filas = netflix.loc[2:4]
rango_filas

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246


In [84]:
# Usando loc , obten los 10 títulos más antiguos en Netflix
# Combinamos sort_values con iloc(Ejemplo base: netflix_ordenado = df_netflix.sort_values("release_year", ascending=True))
print('=== 10 títulos más antiguos en Netflix ===')

# Ordenamos por año
netflix_antiguos = netflix.sort_values(by='release_year', ascending=True)

# Usamos loc con las etiquetas de los primeros 10 registros ordenados
top_10_antiguos = netflix_antiguos.loc[netflix_antiguos.index[:10]]

print('=== 10 títulos más antiguos en Netflix ===')
top_10_antiguos[['title', 'release_year']]

=== 10 títulos más antiguos en Netflix ===
=== 10 títulos más antiguos en Netflix ===


,title,release_year
0,Five Came Back: The Reference Films,1945
27,The Blazing Sun,1954
9,White Christmas,1954
26,Dark Waters,1956
12,Cairo Station,1958
22,Professor,1962
24,Saladin the Victorious,1963
19,Amrapali,1966
15,Prince,1969
7,Monty Python's Flying Circus,1969


---
---
## PARTE C: Procesamiento de Texto — *Dracula* (Bram Stoker)

En esta sección crear las cajas pertinentes a fin de realizar las siguientes actividades sobre el libro de Drácula:
1. Descargar texto desde una URL con `urllib`, crear una variable time, que contabilize el tiempo desde que se inicia la descarga.
2. Eliminar encabezados y pies de página propios de Project Gutenberg.
3. Eliminar signos de puntuación, números y caracteres especiales con `re`.
4. Procesar texto: minúsculas, tokenizar, filtrar.
5. Contar frecuencias con `collections.Counter`.
6. Mostrar el top 20 de las palabras mas frecuentes.
7. Guardar resultados en un archivo de texto(txt), donde se muestre el top 20 de palabras mas frecuentes y su cantidad de incidencias, asi como el tiempo total demorado desde que se descarga el libro, hasta que se cuenta las palabras mas frecuentes.

Usaremos **"Dracula"** de Bram Stoker, disponible en [Project Gutenberg](https://www.gutenberg.org/ebooks/345).  
https://www.gutenberg.org/cache/epub/345/pg345.txt



In [87]:
#Desarrollo del estudiante
inicio_tiempo = time.time()
url = "https://www.gutenberg.org/cache/epub/345/pg345.txt"

print("Descargando")
with urllib.request.urlopen(url) as response:
    texto_raw = response.read().decode('utf-8')

Descargando


In [88]:
inicio_libro = texto_raw.find("*** START OF THE PROJECT GUTENBERG EBOOK")
fin_libro = texto_raw.find("*** END OF THE PROJECT GUTENBERG EBOOK")

In [89]:
texto_limpio = texto_raw[inicio_libro:fin_libro]
# Pasamos a minúsculas primero (Paso 4)
texto_procesado = texto_limpio.lower()
# Reemplazamos todo lo que NO sea una letra (a-z) por un espacio
texto_procesado = re.sub(r'[^a-z\s]', ' ', texto_procesado)

In [90]:
palabras = texto_procesado.split()

In [91]:
conteo = Counter(palabras)

In [93]:
top_20 = conteo.most_common(20)

tiempo_total = time.time() - inicio_tiempo

print(f"¡Procesamiento terminado en {tiempo_total:.4f} segundos!")
print("\nTop 20 palabras más frecuentes:")
for palabra, frec in top_20:
    print(f"{palabra}: {frec}")

¡Procesamiento terminado en 33.8361 segundos!

Top 20 palabras más frecuentes:
the: 7916
and: 5907
i: 4844
to: 4664
of: 3633
a: 2957
he: 2580
in: 2511
that: 2488
it: 2175
was: 1881
as: 1591
we: 1552
for: 1538
is: 1505
his: 1472
me: 1457
you: 1413
not: 1401
with: 1287
